# Task 3: Symbolic Logic and Reasoning Basics

Translate small claims into symbols, distinguish truth from inference, and inspect a rule-based explanation before using LLMs or external solvers.

## How to use this notebook
Run cells in order using Python 3. Only the Python standard library is needed; no GPU, simulator, API key, or paid service is required.

For **Predict** questions, write your answer before running the example. Wrong predictions are fine: explain what changed in your understanding. For **Exercises**, replace `None` or `pass` with your solution, then run the checks.

## Submission
Upload the completed notebook with outputs and written responses to Canvas. Restart the kernel and run all cells before submitting. Include a short reflection: what did you learn, what was difficult, and what question remains?

## 1. Propositions and implication
A proposition has a truth value. `and`, `or`, and `not` combine propositions. Material implication P → Q is equivalent to `(not P) or Q`; it is false only when P is true and Q is false. It does not establish causation.

**Predict:** What is the value when P is false and Q is false?

**My prediction:** True. An implication is only false when P is true and Q is false, so F → F is "vacuously" true.
**Result:** Correct — the first printed row is `False False True`.

In [1]:
for p in (False, True):
    for q in (False, True):
        print(p, q, (not p) or q)

False False True
False True True
True False False
True True True


### Exercise 1
Implement implication and logical equivalence. Explain why P → Q does not allow you to infer Q → P.

In [2]:
def implies(p, q):
    """Material implication: false only when p is true and q is false."""
    return (not p) or q

def equivalent(p, q):
    """Biconditional: p -> q and q -> p."""
    return implies(p, q) and implies(q, p)

In [3]:
for p in (False, True):
    for q in (False, True):
        assert implies(p, q) == ((not p) or q)
        assert equivalent(p, q) == (p == q)
print("Checks passed")

Checks passed


**Why P → Q does not let you infer Q → P:** the assignment P = False, Q = True makes P → Q true but Q → P false. One true implication says nothing about the reverse direction; "if it's raining the ground is wet" doesn't mean "if the ground is wet it's raining" (a sprinkler could have done it).

## 2. Translating language and finding counterexamples
A rule “if registered, then has access” becomes R → A. A satisfying assignment makes every premise true. A conclusion follows only if every such assignment also makes the conclusion true.

**Predict:** If someone has access, must they be registered?

**My prediction:** No. R → A only says registration guarantees access; someone could have access some other way.
**Result:** Correct — `(False, True)` is one of the models: not registered, but has access.

In [4]:
assignments = [(r, a) for r in (False, True) for a in (False, True)]
models = [(r, a) for r, a in assignments if implies(r, a)]
print(models)

[(False, False), (False, True), (True, True)]


### Exercise 2
Find all assignments that satisfy R → A and A but make R false. Then find assignments satisfying R → A and R but making A false. Explain the difference between affirming the consequent and modus ponens.

In [5]:
# Premises R -> A and A hold, but R is false (affirming the consequent)
converse_counterexamples = [(r, a) for r, a in assignments
                            if implies(r, a) and a and not r]

# Premises R -> A and R hold, but A is false (modus ponens)
modus_ponens_counterexamples = [(r, a) for r, a in assignments
                                if implies(r, a) and r and not a]

print("converse counterexamples:", converse_counterexamples)
print("modus ponens counterexamples:", modus_ponens_counterexamples)

converse counterexamples: [(False, True)]
modus ponens counterexamples: []


In [6]:
assert converse_counterexamples == [(False, True)]
assert modus_ponens_counterexamples == []
print("Checks passed")

Checks passed


**Affirming the consequent vs. modus ponens:** Modus ponens (R → A, R, therefore A) is valid: there is no assignment where both premises are true and A is false, so the counterexample list is empty. Affirming the consequent (R → A, A, therefore R) is invalid: `(False, True)` satisfies both premises but makes R false. Having access is consistent with not being registered, so we can't conclude registration from access.

## 3. Facts, rules, and forward chaining
A small knowledge base contains named facts and rules. A rule fires when all its premises are known. Repeatedly apply rules until no facts are added: this is a fixed point. These rules only add positive facts; failure to derive a fact does not establish its negation.

**Predict:** Can the second rule fire before the first?

**My prediction:** Not on the first pass if the rules are checked in list order, because `eligible_lee` isn't known yet. If the list were reversed, the access rule would be checked first and skipped, so a single pass would miss `access_lee`. That's why we loop until nothing new is added.
**Result:** Correct — this is what the reversed-order check in Exercise 3 tests.

In [7]:
facts = {"registered_lee", "completed_training_lee"}
rules = [( {"registered_lee", "completed_training_lee"}, "eligible_lee"),
         ( {"eligible_lee"}, "access_lee") ]
for premises, conclusion in rules:
    print(premises, "->", conclusion)

{'completed_training_lee', 'registered_lee'} -> eligible_lee
{'eligible_lee'} -> access_lee


### Exercise 3
Implement `infer(facts, rules)` to return the closure without modifying the input facts. Test rules in reverse order too; one pass is insufficient in general.

In [8]:
def infer(facts, rules):
    """Forward chaining to a fixed point. Returns a new set; the input is not modified."""
    known = set(facts)
    changed = True
    while changed:
        changed = False
        for premises, conclusion in rules:
            if premises <= known and conclusion not in known:
                known.add(conclusion)
                changed = True
    return known

In [9]:
expected = facts | {"eligible_lee", "access_lee"}
assert infer(facts, rules) == expected
assert infer(facts, list(reversed(rules))) == expected
assert facts == {"registered_lee", "completed_training_lee"}
assert infer({"a"}, [({"a"}, "b"), ({"b"}, "a")]) == {"a", "b"}
assert "access_sam" not in infer(facts, rules)
print("Checks passed")

Checks passed


## 4. Explanations and human decisions
An inference trace records which rule produced a conclusion. A valid derivation depends on the chosen premises; it does not establish that the premises are accurate or the policy is fair.

**Predict:** What happens if the training record is missing?

**My prediction:** Neither rule fires, so the output is just `{'registered_lee'}`. Lee is not *shown* to be eligible, but that doesn't prove Lee is ineligible.
**Result:** Correct.

In [10]:
print(infer({"registered_lee"}, rules))

{'registered_lee'}


### Exercise 4
Implement `explain_access(known)` returning a dictionary with keys `derived` (Boolean), `missing` (set of required input facts), and `trace` (list of derived facts in rule order). This helper is specific to the two rules above. Then write 200–300 words: translate the policy into plain language, describe a person harmed by a missing record, and propose a way for that person to contest it. Distinguish “not derived” from “false.”

In [11]:
def explain_access(known):
    """Explain whether access_lee is derived under the two rules above."""
    required = {"registered_lee", "completed_training_lee"}
    trace = []
    current = set(known)
    # Rule 1: registered + completed training -> eligible
    if required <= current:
        current.add("eligible_lee")
        trace.append("eligible_lee")
    # Rule 2: eligible -> access
    if "eligible_lee" in current:
        current.add("access_lee")
        trace.append("access_lee")
    derived = "access_lee" in current
    missing = set() if derived else required - set(known)
    return {"derived": derived, "missing": missing, "trace": trace}

In [12]:
assert explain_access(facts) == {"derived": True, "missing": set(), "trace": ["eligible_lee", "access_lee"]}
assert explain_access({"registered_lee"}) == {"derived": False, "missing": {"completed_training_lee"}, "trace": []}
assert explain_access(set())["missing"] == facts
print("Checks passed")

Checks passed


### Exercise 4 written response

**The policy in plain language:** A person gets access to the system if they are eligible, and they are eligible if they are registered *and* have completed training. Both records have to be present; neither one alone is enough.

**A person harmed by a missing record:** Suppose Lee registered and finished the training session, but the trainer forgot to upload the attendance sheet, so `completed_training_lee` is never entered. The system runs the rules, neither rule fires, and Lee is shown "no access." From Lee's side it looks like a decision was made about them, but the system never actually decided that Lee *didn't* complete training. It only failed to find a record that they did. If Lee needs access to do their job or finish a class, they are blocked by a data-entry mistake that the explanation hides unless it lists what is missing.

**"Not derived" vs. "false":** These rules only add positive facts, so the system can never prove `access_lee` is false. It can only say access was not derived from what it knows. That is the closed-world assumption: treating "I couldn't find it" as "it isn't true." The result should be reported as "access not granted: missing completed_training_lee," not "Lee is ineligible."

**How Lee could contest it:** The `explain_access` output already gives the pieces needed: `derived = False` and `missing = {completed_training_lee}`. The system should show this to Lee, along with a way to submit evidence (for example a training certificate or the trainer's confirmation). A person reviews it, adds the fact, and the rules are re-run. The trace then shows exactly which rule changed the outcome, so the fix can be checked.

## Reflection

**Your response:**

The main thing I learned is the difference between something being *true* and something being *derivable*. The truth-table section made implication click: P → Q is only false in one row, which is why it says nothing about Q → P, and why modus ponens has zero counterexamples while affirming the consequent has one. Writing the counterexample search as a list comprehension over every assignment made "valid" mean something concrete: no row where all the premises hold and the conclusion fails.

The forward-chaining part was the most interesting. My first version of `infer` made one pass over the rules, and the reversed-order check failed: with the access rule checked first, `access_lee` was skipped because `eligible_lee` hadn't been added yet. I fixed it by looping until a full pass adds nothing new (a fixed point), which also handles the `a → b, b → a` cycle without looping forever because a fact is only added once. I also made sure to copy the input set so the caller's `facts` isn't modified, which one of the checks tests.

What was difficult was the written part of Exercise 4: it's easy to say "Lee is not eligible" when the system has actually only failed to find a record. A question I still have is how systems like Logic-LM or LINC handle this when an LLM writes the facts. If the translation leaves out a fact, the solver will happily report "not derived," and it isn't obvious how a user would know a premise went missing.